# Pain diary cohort

Does treatment arm B change how pain evolves over the first six months? The data is synthetic, made for the Whybook demo.

## Load and reshape

In [1]:
import whybook
import pandas as pd
import statsmodels.formula.api as smf
from examples.pain_diary.prep import MIN_DAYS, drop_sparse, load_diary_raw, load_olink, load_patients, to_long

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [2]:
diary_raw = load_diary_raw()
patients = load_patients()
olink = load_olink()

In [3]:
diary = to_long(diary_raw)
diary.head()

,patient_id,diary_day,week,pain_score,sleep_hours,analgesic_use,cycle_day,bleeding,mood,notes
0,P001,1,1,2.0,7.4,none,17,False,4.0,
1,P001,2,1,2.0,7.5,none,18,False,5.0,
2,P001,3,1,2.0,8.4,none,19,False,4.0,
3,P001,4,1,3.0,7.6,none,20,False,4.0,
4,P001,5,1,2.0,7.3,none,21,False,5.0,


## Shape

In [4]:
weekly = (
    diary.pipe(drop_sparse)
    .merge(patients[["patient_id", "treatment_arm"]], on="patient_id")
    .groupby(["patient_id", "treatment_arm", "week"], as_index=False, observed=True)["pain_score"]
    .mean()
)
whybook.ribbon(weekly, x="week", y="pain_score", by="treatment_arm", ci="bootstrap", n_boot=1000)

<whybook ribbon: pain_score by week>

## Model

In [5]:
model_data = weekly.merge(patients[["patient_id", "age", "site"]], on="patient_id", how="left").assign(
    month=lambda d: (d["week"] - 1) / 4.345
)
lmm_fit = smf.mixedlm(
    "pain_score ~ treatment_arm * month + age",
    data=model_data,
    groups="patient_id",
    re_formula="~month",
    missing="drop",
).fit()
lmm_fit.summary().tables[1]

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,5.009,0.355,14.113,0.000,4.313,5.705
treatment_arm[T.B],-0.857,0.160,-5.359,0.000,-1.170,-0.544
month,-0.750,0.036,-20.595,0.000,-0.821,-0.679
treatment_arm[T.B]:month,-0.318,0.050,-6.383,0.000,-0.416,-0.221
age,-0.003,0.009,-0.281,0.779,-0.021,0.016
patient_id Var,1.751,0.211,,,,
patient_id x month Cov,-0.250,0.051,,,,
month Var,0.152,0.020,,,,


In [7]:
from statsmodels.miscmodels.ordinal_model import OrderedModel

ordinal_data = model_data.assign(
    pain_level=pd.cut(model_data["pain_score"], [-1, 3, 6, 10], labels=["mild", "moderate", "severe"]),
    arm_b=(model_data["treatment_arm"] == "B").astype(int),
)
estimates = []
for i in range(40):
    whybook.progress(i / 40, f"bootstrap {i + 1} of 40")
    sample = ordinal_data.sample(frac=1, replace=True, random_state=i)
    fit = OrderedModel.from_formula("pain_level ~ 0 + arm_b * month + age", sample, distr="logit").fit(method="bfgs", disp=False)
    estimates.append(fit.params["arm_b:month"])
whybook.progress(1.0, "done")
ordinal_arm_month = pd.Series(estimates, name="arm B × month (log odds)").describe()
ordinal_arm_month

count    40.000000
mean     -1.400307
std       0.083718
min      -1.632561
25%      -1.433042
50%      -1.403292
75%      -1.355648
max      -1.216267
Name: arm B × month (log odds), dtype: float64

In [ ]:
# Is cycle_start associated with pain_score in [5b], adjusting for site?
import statsmodels.formula.api as smf

_data = ordinal_data
_data = _data.merge(diary_raw.groupby(['patient_id', 'week'], as_index=False, observed=True)['cycle_start'].mean(), on=['patient_id', 'week'], how="left")
cycle_start_vs_pain_score = smf.ols("pain_score ~ cycle_start + site", data=_data).fit()
del _data
cycle_start_vs_pain_score.summary().tables[1]